# 00 — Guardrails audit

Live invariant dashboard for **G1–G10 + RISK-04** against the current source tree.

**No LLM, no MCP, no network.** Every cell either passes silently or raises.

If a cell fails, the guardrail it covers has regressed — open the source file referenced in the assertion and figure out which structural defense was weakened.

Reference: `README.md#guardrails-g1g10`, `src/hsb/agents/_sdk_options.py`, `src/hsb/agents/risk_agent.py`.

In [ ]:
from _helpers import ensure_src_on_path

ROOT = ensure_src_on_path()
print("repo root =", ROOT)

## G1 — OAuth2 only (function-entry guard)

The guard is **function-entry**, not module-import. A dev environment with `ANTHROPIC_API_KEY` set should not break pytest collection — but a call to `make_options()` should refuse.

We probe both branches: env unset (passes) and env set (raises).

In [ ]:
import importlib
import os

# Importing the chokepoint module must NOT raise even if the env var is set.
os.environ["ANTHROPIC_API_KEY"] = "pretend-this-is-leaked"
from hsb.agents import _sdk_options

importlib.reload(_sdk_options)  # prove a reload is also safe
print("module-import path is clean even with ANTHROPIC_API_KEY set")

In [ ]:
# G1 must raise at function-entry of make_options() / assert_oauth2_only().
from hsb.agents._sdk_options import assert_oauth2_only, make_options

raised = False
try:
    assert_oauth2_only()
except RuntimeError as e:
    raised = True
    assert "G1 violation" in str(e)
assert raised, "G1 did not raise when ANTHROPIC_API_KEY is set"

raised = False
try:
    make_options(permission_mode="dontAsk", allowed_tools=["Read"])
except RuntimeError:
    raised = True
assert raised, "make_options should refuse when ANTHROPIC_API_KEY is set"

del os.environ["ANTHROPIC_API_KEY"]
print("G1: function-entry guard works in both code paths")

In [ ]:
# With the env clean, make_options() should construct without complaint.
opts = make_options(permission_mode="dontAsk", allowed_tools=["Read"])
assert "Read" in (opts.allowed_tools or [])
print("G1: clean env path produces ClaudeAgentOptions OK")

## G2 — `"Agent"` is forbidden in any `allowed_tools`

WORC-02: no sub-subagent dispatch. The factory raises `ValueError` if `"Agent"` slips into `allowed_tools`.

In [ ]:
raised = False
try:
    make_options(permission_mode="dontAsk", allowed_tools=["Read", "Agent"])
except ValueError as e:
    raised = True
    assert "G2 violation" in str(e)
assert raised, "G2 should refuse Agent in allowed_tools"
print("G2: factory rejects Agent in allowed_tools")

In [ ]:
# Belt-and-braces: grep every agent file for an 'Agent' literal in an allow-list.
# We tolerate the FORBIDDEN list itself in _sdk_options.py and any prose strings.
import re

agent_dir = ROOT / "src" / "hsb" / "agents"
violations = []
for p in sorted(agent_dir.glob("*.py")):
    text = p.read_text()
    # Look for an allowed_tools list literal that contains a bare 'Agent' string.
    for m in re.finditer(r"allowed_tools\s*=\s*\[(.*?)\]", text, re.DOTALL):
        body = m.group(1)
        if re.search(r"['\"]Agent['\"]", body):
            violations.append((p.name, body[:120]))
assert not violations, f"G2 grep found Agent in an allowed_tools literal: {violations}"
print("G2: no agent module declares Agent in its allowed_tools")

## G3 — Runtime backstop catches `Task`-tool dispatch

If the SDK ever regresses and bypasses `allowed_tools`, the per-message scan in every receive loop catches a Task block and raises.

In [ ]:
from claude_agent_sdk import AssistantMessage

from hsb.agents._sdk_options import assert_no_task_dispatch


class FakeBlock:
    def __init__(self, name):
        self.name = name


msg = AssistantMessage(content=[FakeBlock("Task")], model="claude-opus-4-7")
raised = False
try:
    assert_no_task_dispatch(msg)
except RuntimeError as e:
    raised = True
    assert "G3 violation" in str(e)
assert raised, "G3 should raise on Task block"
print("G3: AssistantMessage with Task block raises")

In [ ]:
# Innocent messages must NOT raise.
msg = AssistantMessage(
    content=[FakeBlock("Read"), FakeBlock("Bash")], model="claude-opus-4-7"
)
assert_no_task_dispatch(msg)  # no exception
print("G3: clean assistant message passes through silently")

## G4 — Risk Agent skill 14 is structurally air-gapped

The Auto-Improvement Trigger SDK call has `allowed_tools=[]`, no MCP, Haiku model, and a hard budget. Verifying the literals appear in the source — the structural defense, not just the runtime assertion.

In [ ]:
ra = (ROOT / "src" / "hsb" / "agents" / "risk_agent.py").read_text()
assert "allowed_tools=[]" in ra, "G4 layer 1: empty allowed_tools missing"
assert "model='claude-haiku-4-5'" in ra or 'model="claude-haiku-4-5"' in ra, (
    "G4: Haiku pin missing"
)
assert "max_budget_usd=0.05" in ra, "G4: budget cap missing"
assert "max_turns=3" in ra, "G4: max_turns cap missing"
print("G4: skill-14 air-gap config literals all present")

## RISK-04 — 4-layer defense (the strongest milestone-level invariant)

1. STRUCTURAL — skill-14 SDK call (covered by G4 above).
2. PARSE-TIME — `AutoImprovementTrigger.linear_state` is `Literal["suggested"]`.
3. IMPORT-TIME — `risk_agent.py` does NOT import `hsb.agents.linear_agent`.
4. RUNTIME — `linear_write_guard` denies frames originating in `risk_agent.py` (except the operator-delegated path).

In [ ]:
assert "from hsb.agents.linear_agent" not in ra, (
    "RISK-04 layer 3: risk_agent imports linear_agent"
)
assert "import hsb.agents.linear_agent" not in ra, (
    "RISK-04 layer 3: risk_agent imports linear_agent"
)
print("RISK-04 layer 3 (import-time): risk_agent.py does not import linear_agent")

In [ ]:
from pydantic import ValidationError

from hsb.contracts.risk import AutoImprovementTrigger

trig = AutoImprovementTrigger(
    title="t", description="d", pattern_evidence=["LIN-1", "LIN-2"], suggested_type="x"
)
assert trig.linear_state == "suggested"
raised = False
try:
    AutoImprovementTrigger(
        title="t",
        description="d",
        pattern_evidence=["LIN-1", "LIN-2"],
        suggested_type="x",
        linear_state="created",
    )
except ValidationError:
    raised = True
assert raised, "RISK-04 layer 2: parser accepted linear_state != 'suggested'"
print("RISK-04 layer 2 (parse-time): linear_state Literal enforced")

In [ ]:
# Layer 4: the linear_write_guard helper. We exercise it by calling a wrapped
# function from a 'fake risk_agent' frame and confirming PermissionError.

from hsb.agents._sdk_options import linear_write_guard


@linear_write_guard
def fake_write():
    return "ok"


# Direct call from this notebook (no risk_agent.py frame in stack) → allowed.
assert fake_write() == "ok"
print("RISK-04 layer 4: guard allows non-risk callers")

# Simulate a call originating from risk_agent.py by exec-ing in a faked filename.
import os
import tempfile

tmpdir = tempfile.mkdtemp()
fake_path = os.path.join(tmpdir, "hsb", "agents", "risk_agent.py")
os.makedirs(os.path.dirname(fake_path), exist_ok=True)
with open(fake_path, "w") as f:
    f.write("def call_it(fn):\n    return fn()\n")
import importlib.util

spec = importlib.util.spec_from_file_location("hsb.agents.risk_agent_dummy", fake_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
raised = False
try:
    mod.call_it(fake_write)
except PermissionError as e:
    raised = True
    assert "RISK-04" in str(e)
assert raised, "RISK-04 layer 4: guard did not deny risk_agent frame"
print("RISK-04 layer 4: guard denies frames from risk_agent.py")

## G9 — Knowledge Store pre-write hook

`KnowledgeStorageInput.applicability` rejects `"all tasks"`, `"all"`, `"n/a"`, `"tbd"`, and empty/whitespace strings (case-insensitive).

In [ ]:
from hsb.contracts.knowledge import KnowledgeStorageInput

base = dict(
    title="Caching DB writes",
    type="implementation",
    context="ctx",
    evidence={
        "linear_issue": "LIN-1",
        "pr": "#1",
        "files": ["x.py"],
        "qa_finding": "F1",
    },
    insight="ins",
    recommendation="rec",
    date="2026-05-09",
)
rejected = ["all tasks", "All Tasks", "ALL", "n/a", "tbd", "TBD", "", "   ", " "]
for bad in rejected:
    raised = False
    try:
        KnowledgeStorageInput(**base, applicability=bad)
    except Exception:
        raised = True
    assert raised, f"G9 accepted {bad!r}"
ok = KnowledgeStorageInput(
    **base, applicability="Postgres write paths in payments domain"
)
assert ok.applicability.startswith("Postgres")
print("G9: applicability validator rejects all banned values, accepts a real one")

## QA cycle cap (Pitfall 2 / QAAG-04)

Pydantic `model_validator` rejects `qa_cycle_count >= 3` with `qa_status='changes_required'` and requires `tech_debt_annotation` at cap. This is the deterministic last line of defense — the SKILL.md instruction is only probabilistic.

In [ ]:
from hsb.contracts.qa import QAOutput

# Cap reached + still changes_required → must fail
raised = False
try:
    QAOutput(
        work_item_id="LIN-1",
        qa_status="changes_required",
        qa_cycle_count=3,
        summary="s",
        findings=[],
        tech_debt_annotation="t",
    )
except Exception:
    raised = True
assert raised, "QA cap accepted changes_required at cycle 3"

# Cap reached + approved + missing annotation → must fail
raised = False
try:
    QAOutput(
        work_item_id="LIN-1",
        qa_status="approved",
        qa_cycle_count=3,
        summary="s",
        findings=[],
    )
except Exception:
    raised = True
assert raised, "QA cap accepted missing tech_debt_annotation at cycle 3"

# Cap reached + approved + annotation present → OK
ok = QAOutput(
    work_item_id="LIN-1",
    qa_status="approved",
    qa_cycle_count=3,
    summary="s",
    findings=[],
    tech_debt_annotation="leaving X for follow-up",
)
assert ok.qa_cycle_count == 3
print("QA cycle cap: validator rejects both runaway shapes, accepts the canonical one")

## Smoke: every agent module imports cleanly

If a structural change breaks a top-level import, every other notebook will misreport. Cheap canary.

In [ ]:
import importlib

modules = [
    "hsb.agents._sdk_options",
    "hsb.agents.hooks",
    "hsb.agents.linear_agent",
    "hsb.agents.backlog_agent",
    "hsb.agents.builder_agent",
    "hsb.agents.git_agent",
    "hsb.agents.qa_agent",
    "hsb.agents.work_item_orchestrator",
    "hsb.agents.global_orchestrator",
    "hsb.agents.main_orchestrator",
    "hsb.agents.uat_agent",
    "hsb.agents.risk_agent",
    "hsb.agents.intelligence_agent",
    "hsb.contracts.qa",
    "hsb.contracts.knowledge",
    "hsb.contracts.risk",
    "hsb.contracts.uat",
    "hsb.contracts.global_orchestrator",
    "hsb.contracts.main_orchestrator",
    "hsb.contracts.orchestrator",
]
for m in modules:
    importlib.import_module(m)
print(f"OK — all {len(modules)} modules import cleanly")